###  Connecting to Neo4j

In [3]:
from neo4j import GraphDatabase
import pandas as pd

URI = "neo4j+s://8c7b06ba.databases.neo4j.io"
USERNAME = "8c7b06ba"
PASSWORD = "qD0ehEZuPfSVjJ5F6oNsxIrF297R3ckAEmIM0DZ296Y"

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

### Helper to run queries

In [4]:
def run_query(query, params=None):
    with driver.session() as session:
        result = session.run(query, params or {})
        return [record.data() for record in result]

def run_df(query, params=None):
    return pd.DataFrame(run_query(query, params))

### Loading data files

In [5]:
base_path = "/Users/kalafejzo/mids205_project3_kala/data"

users = pd.read_csv(f"{base_path}/users.csv")
friend_edges = pd.read_csv(f"{base_path}/friend_edges.csv")
posts = pd.read_csv(f"{base_path}/posts.csv")
likes = pd.read_csv(f"{base_path}/likes.csv")
circles = pd.read_csv(f"{base_path}/all_circles.csv")

### Loading users

In [6]:
def load_users(tx, rows):
    query = """
    UNWIND $rows AS row
    MERGE (u:User {user_id: toInteger(row.user_id)})
    SET u.is_ego = CASE
      WHEN row.is_ego IN [True, 'True', 'true', 1, '1'] THEN true
      ELSE false
    END,
    u.degree = toInteger(row.degree),
    u.circle_count = toInteger(row.circle_count),
    u.ego_network_count = toInteger(row.ego_network_count),
    u.primary_ego_id = CASE
      WHEN row.primary_ego_id IS NULL OR row.primary_ego_id = '' THEN NULL
      ELSE toInteger(row.primary_ego_id)
    END
    """
    tx.run(query, rows=rows)

with driver.session() as session:
    session.execute_write(load_users, users.to_dict("records"))

KeyboardInterrupt: 

### Loading Friendships


In [ ]:
def load_friend_edges(tx, rows):
    query = """
    UNWIND $rows AS row
    MATCH (u1:User {user_id: toInteger(row.source)})
    MATCH (u2:User {user_id: toInteger(row.target)})
    MERGE (u1)-[:FRIENDS_WITH]->(u2)
    """
    tx.run(query, rows=rows)

with driver.session() as session:
    session.execute_write(load_friend_edges, friend_edges.to_dict("records"))

### Loading Posts

In [ ]:
def load_posts(tx, rows):
    query = """
    UNWIND $rows AS row
    MATCH (u:User {user_id: toInteger(row.user_id)})
    MERGE (p:Post {post_id: row.post_id})
    SET p.topic_circle = row.topic_circle,
        p.engagement_seed = toFloat(row.engagement_seed)
    MERGE (u)-[:POSTED]->(p)
    """
    tx.run(query, rows=rows)

with driver.session() as session:
    session.execute_write(load_posts, posts.to_dict("records"))

### Loading Likes

In [ ]:
def load_likes(tx, rows):
    query = """
    UNWIND $rows AS row
    MATCH (u:User {user_id: toInteger(row.user_id)})
    MATCH (p:Post {post_id: row.post_id})
    MERGE (u)-[:LIKED]->(p)
    """
    tx.run(query, rows=rows)

with driver.session() as session:
    session.execute_write(load_likes, likes.to_dict("records"))

### Loading circles

In [ ]:
def load_circles(tx, rows):
    query = """
    UNWIND $rows AS row
    MERGE (c:Circle {circle_key: toString(row.ego_id) + '_' + row.circle_name})
    SET c.ego_id = toInteger(row.ego_id),
        c.circle_name = row.circle_name
    """
    tx.run(query, rows=rows)

def load_circle_memberships(tx, rows):
    query = """
    UNWIND $rows AS row
    MATCH (u:User {user_id: toInteger(row.member_id)})
    MATCH (c:Circle {circle_key: toString(row.ego_id) + '_' + row.circle_name})
    MERGE (u)-[:MEMBER_OF]->(c)
    """
    tx.run(query, rows=rows)

### Loads in batches

In [ ]:
def batch_records(df, batch_size=1000):
    for start in range(0, len(df), batch_size):
        yield df.iloc[start:start+batch_size].to_dict("records")

with driver.session() as session:
    for batch in batch_records(circles, 1000):
        session.execute_write(load_circles, batch)

with driver.session() as session:
    for batch in batch_records(circles, 1000):
        session.execute_write(load_circle_memberships, batch)